In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS swiggy.bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS swiggy.silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS swiggy.gold")
print("✅ Schemas created")

In [0]:
dbutils.widgets.text("base_path", "/Volumes/swiggy/pipeline/raw_data")

In [0]:
BASE_PATH = dbutils.widgets.get("base_path")

LANDING_RESTAURANTS = BASE_PATH + "/landing/restaurants"
LANDING_CUSTOMERS = BASE_PATH + "/landing/customers"
BRONZE_RESTAURANTS = "swiggy.bronze.restaurants"
BRONZE_CUSTOMERS = "swiggy.bronze.customers"

print(f"BASE_PATH: {BASE_PATH}")
print(f"LANDING_REASTURANTS: {LANDING_RESTAURANTS}")
print(f"BRONZE_REASTURANTS: {BRONZE_RESTAURANTS}")
print(f"LANDING_CUSTOMERS: {LANDING_CUSTOMERS}")
print(f"BRONZE_CUSTOMERS: {BRONZE_CUSTOMERS}")



In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, BooleanType
restaurant_schema  =  StructType([
    StructField("restaurant_id", StringType(), True),
    StructField("restaurant_name", StringType(), True),
    StructField("cuisine_type", StringType(), True),
    StructField("avg_rating", DoubleType(), True),
    StructField("avg_preparation_time" , IntegerType(), True),
    StructField("is_active", BooleanType(), True),
    StructField("commission_pct", IntegerType(), True),
    StructField("zone", StringType(), True),
    StructField("restaurant_city", StringType(), True),
    StructField("restaurant_state", StringType(), True),
    StructField("created_at", StringType(), True)
])

customers_schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("customer_name", StringType(), True),
    StructField("customer_email", StringType(), True),
    StructField("customer_phone", StringType(), True),
    StructField("zone", StringType(), True),
    StructField("customer_city", StringType(), True),
    StructField("customer_state", StringType(), True),
    StructField("signup_date", StringType(), True),
    StructField("subscription_tier", StringType(), True),
    StructField("total_orders", IntegerType(), True),
    StructField("preferred_cuisine", StringType(), True),
    StructField("last_order_date", StringType(), True),
    StructField("rfm_segment", StringType(), True),
    StructField("created_at", StringType(), True)
])

In [0]:
restaurant_raw_df = (spark.read
           .format("json")
           .schema(restaurant_schema)
           .load(LANDING_RESTAURANTS))

(restaurant_raw_df.write
      .format("delta")
      .mode("overwrite")
      .saveAsTable(BRONZE_RESTAURANTS))

customer_raw_df = (spark.read
           .format("json")
           .schema(customers_schema)
           .load(LANDING_CUSTOMERS))

(customer_raw_df.write
      .format("delta")
      .mode("overwrite")
      .saveAsTable(BRONZE_CUSTOMERS))


In [0]:
spark.table("swiggy.bronze.restaurants").count()
display(spark.table("swiggy.bronze.restaurants"))
spark.table("swiggy.bronze.customers").count()
display(spark.table("swiggy.bronze.customers"))